<center>
    <img src="https://pyvoa.org/wp-content/uploads/2024/07/logo-pyvoa-1030x325.png" height="75px" alt="Logo Pyvoa" /> 
</center>

# Pyvoa Front with INSEE
Tristan Beau - june 2026

This notebook demonstrates the ability of pyvoa to use external data. The french INSEE (https://www.insee.fr/fr/accueil) provides all deaths informations, but in a specific format. We create here a pandas compatible with pyvoa, and use pyvoa to plot data.

In [1]:
# Run once on Colab / Binder, or on a fresh environment.
# %pip install pyvoa-full

In [4]:
from IPython.display import Markdown
import random
import pyvoa.front as pf
pf.listvis()

['bokeh', 'matplotlib']

## INSEE parsing (from previous pyvoa/pycoa/cocoa versions)

In [5]:
import os
from pathlib import Path
from getpass import getuser
from urllib.parse import urlparse
import requests 
from zlib import crc32
import pandas as pd
from bs4 import BeautifulSoup
import json

import pyvoa.tools as pt

pf.set_verbose_mode(2)

# https://www.data.gouv.fr/datasets/fichier-des-personnes-decedees
url = "https://www.data.gouv.fr/api/1/datasets/fichier-des-personnes-decedees/rdf.jsonld"

maintmpdir=os.path.join(Path.home(),".cache")
tmpdir=os.path.join(maintmpdir,"pyvoa.data"+"_"+getuser())

def get_local_from_actual_url(url,go=False):
    # False : do not reload if exists. True : reload the file.
    if not os.path.exists(tmpdir):
        os.makedirs(tmpdir)
    local_base_filename=urlparse(url).netloc+"_"+str(crc32(bytes(url,'utf-8')))
    local_tmp_filename=os.path.join(tmpdir,local_base_filename)
    local_file_exists=False
    
    if not os.path.exists(local_tmp_filename) or go:
        print("===> downloading "+url+" ...")
        #headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
        headers = {'User-Agent': 'Wget/1.16.3 (darwin14.3.0)'}
        urlfile = requests.get(url, allow_redirects=True,headers=headers) # adding headers for server which does not accept no browser presentation
        fp=open(local_tmp_filename,'wb')
        fp.write(urlfile.content)
        fp.close()
    else:
        print("===> data already available locally, no download.")
        
    print("===> data available locally at : "+local_tmp_filename+" .")
    
    return local_tmp_filename

In [6]:
import requests
import re
from collections import defaultdict

url = "https://www.data.gouv.fr/api/1/datasets/fichier-des-personnes-decedees/rdf.jsonld"
with open(get_local_from_actual_url(url,go=True)) as fp:
    data = json.loads(fp.read())
    
brut = defaultdict(lambda: {'annuel': [], 'mensuel': [], 'trimestriel': []})
for d in data['@graph']:
    if 'title' in d and 'accessURL' in d:   # adapter le nom du champ si nécessaire
        title = d['title']
        access = d['accessURL']             # ← champ URL
        match = re.search(r'\d{4}', title)
        if not match:
            continue
        annee = match.group()
        t = title.lower()
        if re.search(r'-m\d', t):
            brut[annee]['mensuel'].append(access)
        elif re.search(r'-t\d', t):
            brut[annee]['trimestriel'].append(access)
        elif re.search(r'deces-',t):
            brut[annee]['annuel'].append(access)

insee_par_annee = {
    annee: sorted(types['annuel'] if types['annuel'] else types['mensuel'])
    for annee, types in sorted(brut.items())
}

===> downloading https://www.data.gouv.fr/api/1/datasets/fichier-des-personnes-decedees/rdf.jsonld ...
===> data available locally at : /home/beau/.cache/pyvoa.data_beau/www.data.gouv.fr_2877423176 .


In [7]:
import datetime
current_year=datetime.date.today().year
current_month=datetime.date.today().month

y0=2000
y1=2026

In [8]:
import chardet
dc={}
for y in range(y0,y1+1):
    i=str(y) #  in string
    print(i)
    m=0
    for url in insee_par_annee[i]:
        idx=i+'-'+str(m)
        m=m+1
        print(url)
        with open(get_local_from_actual_url(url),'rb') as f:
            raw = f.read()

        encoding = chardet.detect(raw)['encoding']
        if encoding=='ascii':
            encoding='latin1'
        dc.update({idx:raw.decode('latin1').splitlines(keepends=True)})

2000
https://www.data.gouv.fr/api/1/datasets/r/01bd668a-a8ba-4287-838d-3acbd3b30ba2
===> data already available locally, no download.
===> data available locally at : /home/beau/.cache/pyvoa.data_beau/www.data.gouv.fr_2232097930 .
2001
https://www.data.gouv.fr/api/1/datasets/r/da1c8b63-3c0f-4aa7-a244-6fed6b2e3cf8
===> data already available locally, no download.
===> data available locally at : /home/beau/.cache/pyvoa.data_beau/www.data.gouv.fr_2476395760 .
2002
https://www.data.gouv.fr/api/1/datasets/r/8a9ff686-ae72-4cbd-b883-a9e4690c2d48
===> data already available locally, no download.
===> data available locally at : /home/beau/.cache/pyvoa.data_beau/www.data.gouv.fr_1343692326 .
2003
https://www.data.gouv.fr/api/1/datasets/r/edd5725f-2362-49b6-901f-1b43eac5824e
===> data already available locally, no download.
===> data available locally at : /home/beau/.cache/pyvoa.data_beau/www.data.gouv.fr_300709777 .
2004
https://www.data.gouv.fr/api/1/datasets/r/36653f18-ae52-40c8-9c5f-6de43e

In [9]:
def string_to_date(s):
   date=None
   y=int(s[0:4])
   m=int(s[4:6])
   d=int(s[6:8])
   if m==0:
       m=1
   if d==0:
       d=1
   if y==0:
       raise ValueError
   try:
       date=datetime.date(y,m,d)
   except:
       if m==2 and d==29:
           d=28
           date=datetime.date(y,m,d)
           raise ValueError
   return date

In [10]:
list(insee_par_annee.keys())[0]

'1970'

In [11]:
pdict={}
insee_pd=pd.DataFrame()
since_year=int(list(insee_par_annee.keys())[0])

for i in list(dc.keys()):
   print(i)
   data=[]

   for l in dc[i]:
       try:
           [last_name,first_name]=(l[0:80].split("/")[0]).split("*")
           sex=int(l[80])
           birthlocationcode=l[89:94]
           birthlocationname=l[94:124].rstrip()
           birthdate=string_to_date(l[81:89])
           deathdate=string_to_date(l[154:].strip()[0:8]) # sometimes, heading space
           lbis=list(l[154:].strip()[0:8])
           lbis[0:4]=list('2003')
           lbis=''.join(lbis)
           deathdatebis=string_to_date(lbis)
       except ValueError:
           if lbis!='20030229':
               print('Problem in a date parsing insee data for : ',l,lbis)
       deathlocationcode=l[162:167]
       deathlocationshortcode=l[162:164]
       deathid=l[167:176]
       data.append([deathlocationshortcode,deathdate])
   p=pd.DataFrame(data)
   p.columns=['where','death_date']
   insee_pd=pd.concat([insee_pd,p])
insee_pd = insee_pd[['where','death_date']].reset_index(drop=True)
insee_pd = insee_pd.rename(columns={'death_date':'date'})
insee_pd['date']=pd.to_datetime(insee_pd['date']).dt.date
insee_pd['where']=insee_pd['where'].astype(str)
insee_pd = insee_pd.groupby(['date','where']).size().reset_index(name='daily_number_of_deaths')

since_date=str(since_year)+'-01-01'
insee_pd = insee_pd[insee_pd.date>=datetime.date.fromisoformat(since_date)].reset_index(drop=True)
insee_pd['tot_deaths_since_'+since_date]=insee_pd.groupby('where')['daily_number_of_deaths'].cumsum()
insee_pd=insee_pd.drop(columns='daily_number_of_deaths')

2000-0
2001-0
2002-0
Problem in a date parsing insee data for :  CACI*AMALIA/                                                                    21913022999125VUNO                          ALBANIE                       20000103431771        N             00000000
 20030928
2003-0
2004-0
2005-0
2006-0
Problem in a date parsing insee data for :  CARRARD*DANIEL FERNAND/                                                         11938101352427ROBERT MAGNY LANEUVILLE  20031214
Problem in a date parsing insee data for :   REMY                              20061212514542745     N             00000000
 20031214
2007-0
Problem in a date parsing insee data for :  LEITE DA SILVA*JOSE ALEXANDRE/                                                  11957022099139TRADE P 20030709
Problem in a date parsing insee data for :  VOA DE LANHOSO       PORTUGAL                      200707119207885       N             00000000
 20030709
2008-0
2009-0
Problem in a date parsing insee data for :  NERANDJAN*CALYPSO/ 

In [12]:
insee_pd['dc']=insee_pd['tot_deaths_since_1970-01-01']

In [13]:
pf.setvis('bokeh')

In [17]:
pf.plot(input=insee_pd,which='dc',option='sumall',when='01/01/2020:15/07/2026',what='daily',typeofplot='yearly',title='Décès en France enregistrés par l\'INSEE')

Init of AllVisu() with db=in-house data
Memory usage of all columns: 1,069,956 bytes


In [18]:
pf.plot(input=insee_pd,which='dc',option='sumall',when='01/01/2000:31/12/2003',what='daily',typeofplot='yearly',title='Décès en France enregistrés par l\'INSEE')

Init of AllVisu() with db=in-house data
Memory usage of all columns: 654,660 bytes
